### 1. Prereq

In [ ]:
pip install -qU langchain langchain-pinecone langchain-groq "pinecone[grpc]" langchain-community


## 2. Indexing Your Data in Pinecone

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec


In [ ]:
# Load and Split Documents
with open("data.txt", "w") as f:
    f.write("The history of artificial intelligence (AI) began in antiquity, with myths, stories and rumors of artificial beings endowed with intelligence or consciousness by master craftsmen. The seeds of modern AI were planted by classical philosophers who attempted to describe the process of human thinking as the mechanical manipulation of symbols.")

loader = TextLoader("./sample_book_data.txt")
documents = loader.load()

# Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.split_documents(documents)



In [ ]:
# Initialize Embeddings and Pinecone 
# Using a popular open-source model for embeddings
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
# Initialize Pinecone
pc = Pinecone()
index_name = "langchain-groq-rag-demo"

# Check if the index exists, if not, create it
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # The dimension of the all-MiniLM-L6-v2 model
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'
        )
    )

# Create and Populate the Vector Store 
# This will embed your documents and upload them to the Pinecone index
vectorstore = PineconeVectorStore.from_documents(docs, embeddings_model, index_name=index_name)

print("Data has been successfully indexed in Pinecone.")

### Building the RAG Chain with LangChain and Groq

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq



In [ ]:
# Initialize the Groq LLM 

# Using a fast model from Groq
llm = ChatGroq(temperature=0, model_name="llama3-8b-8192")

# Create a Retriever 
# The retriever fetches relevant documents from the vector store
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

# Define the Prompt Template 
# This template structures the input for the LLM
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Create the RAG Chain ---
# This chain links the retriever, prompt, LLM, and output parser
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("\nRAG Chain with Groq and Pinecone is ready.")

### Running the RAG Chain to Get Answers

In [ ]:
# --- Ask a question ---
question = "What is the history of AI?"
response = rag_chain.invoke(question)

print(f"\nQuestion: {question}")
print(f"Answer: {response}")

# Example of asking something not in the context
question_2 = "What is the capital of France?"
response_2 = rag_chain.invoke(question_2)
print(f"\nQuestion: {question_2}")
print(f"Answer: {response_2}")
